# Main-Sweep Statistical-Rigor Revision: 20 Seeds + Confidence Intervals Throughout

This notebook re-runs every one of those sweeps at **20 seeds instead of 3** (`configs/*_v2.yaml`, otherwise identical to the originals -- same methods, same dataset parameters, same severity levels), attaches a 95% seed-cluster bootstrap CI to every accuracy-drop number (`ssl_spatial.metrics.bootstrap.bootstrap_ci_drop`, the same resampling scheme already validated against `changepoint_analysis.py`'s existing breakpoint CI), and re-derives Table 2's correlations with the same CI treatment (`ssl_spatial.experiments.localized_mmd_analysis_v2`). Point estimates are compared directly against the original 3-seed numbers throughout, since several of them move enough to matter.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))

import pandas as pd

from ssl_spatial.metrics.bootstrap import bootstrap_ci_drop

pd.set_option("display.width", 120)
print("Repo root:", REPO_ROOT)

Repo root: /mnt/c/Users/Lenovo/Desktop/SSL


## 1. Synthetic main sweep (Table 1), 20 seeds

`configs/controlled_experiment_v2.yaml`: identical to `controlled_experiment.yaml` except 3 → 20 seeds (2,160 rows instead of 324).

In [2]:
synth = pd.read_csv(REPO_ROOT / "results" / "controlled_experiment_v2.csv")
drop_synth = bootstrap_ci_drop(synth, group_cols=["method"], value_col="accuracy_out_region",
                                alpha_col="mismatch_alpha", n_boot=200, seed=0)
drop_synth["drop_pp"] = (drop_synth["drop"] * 100).round(2)
drop_synth["95% CI (pp)"] = drop_synth.apply(
    lambda r: f"[{r['ci_low']*100:.2f}, {r['ci_high']*100:.2f}]", axis=1)
drop_synth["drop_pp (original, 3 seeds)"] = drop_synth["method"].map(
    {"self_training": 8.8, "supervised_only": 6.3, "label_propagation": 4.9})
drop_synth.sort_values("drop_pp", ascending=False)[
    ["method", "drop_pp", "95% CI (pp)", "drop_pp (original, 3 seeds)"]].reset_index(drop=True)

,method,drop_pp,95% CI (pp),"drop_pp (original, 3 seeds)"
0,supervised_only,6.16,"[4.54, 7.83]",6.3
1,self_training,5.60,"[3.86, 7.37]",8.8
2,label_propagation,4.15,"[2.49, 5.81]",4.9


## 2. PovertyMap-WILDS (Table 4), 20 seeds

`configs/wilds_benchmark_v2.yaml`: identical except 3 → 20 seeds (360 rows instead of 54).

In [3]:
wilds = pd.read_csv(REPO_ROOT / "results" / "wilds_povertymap_experiment_v2.csv")
drop_wilds = bootstrap_ci_drop(wilds, group_cols=["method"], value_col="accuracy_out_region",
                                alpha_col="mismatch_alpha", n_boot=200, seed=0)
drop_wilds["drop_pp"] = (drop_wilds["drop"] * 100).round(2)
drop_wilds["95% CI (pp)"] = drop_wilds.apply(
    lambda r: f"[{r['ci_low']*100:.2f}, {r['ci_high']*100:.2f}]", axis=1)
drop_wilds["drop_pp (original, 3 seeds)"] = drop_wilds["method"].map(
    {"self_training": 15.3, "label_propagation": 10.2, "supervised_only": 5.1})
drop_wilds.sort_values("drop_pp", ascending=False)[
    ["method", "drop_pp", "95% CI (pp)", "drop_pp (original, 3 seeds)"]].reset_index(drop=True)

,method,drop_pp,95% CI (pp),"drop_pp (original, 3 seeds)"
0,self_training,14.57,"[9.72, 21.40]",15.3
1,label_propagation,12.10,"[10.13, 14.07]",10.2
2,supervised_only,10.17,"[6.90, 13.74]",5.1


## 3. Real-world datasets (Table 5), 20 seeds

`configs/{housing,socioeconomic,air_quality}_benchmark_v2.yaml`: identical except 3 → 20 seeds (480 rows each instead of 72).

In [4]:
ORIGINAL_REAL_WORLD = {
    ("housing", "supervised_only"): 11.9, ("housing", "self_training"): 10.3,
    ("housing", "label_propagation"): 2.9, ("housing", "reweighted_self_training"): 9.7,
    ("socioeconomic", "supervised_only"): -1.8, ("socioeconomic", "self_training"): -1.5,
    ("socioeconomic", "label_propagation"): -1.3, ("socioeconomic", "reweighted_self_training"): -2.3,
    ("air_quality", "supervised_only"): 18.1, ("air_quality", "self_training"): 17.1,
    ("air_quality", "label_propagation"): 29.0, ("air_quality", "reweighted_self_training"): 21.0,
}

real_world_tables = {}
for name in ["housing", "socioeconomic", "air_quality"]:
    df = pd.read_csv(REPO_ROOT / "results" / f"{name}_experiment_v2.csv")
    d = bootstrap_ci_drop(df, group_cols=["method"], value_col="accuracy_out_region",
                           alpha_col="mismatch_alpha", n_boot=200, seed=0)
    d["drop_pp"] = (d["drop"] * 100).round(2)
    d["95% CI (pp)"] = d.apply(lambda r: f"[{r['ci_low']*100:.2f}, {r['ci_high']*100:.2f}]", axis=1)
    d["drop_pp (original, 3 seeds)"] = d["method"].map(lambda m: ORIGINAL_REAL_WORLD[(name, m)])
    real_world_tables[name] = d.sort_values("drop_pp", ascending=False)[
        ["method", "drop_pp", "95% CI (pp)", "drop_pp (original, 3 seeds)"]].reset_index(drop=True)

print("--- Housing ---"); display(real_world_tables["housing"])
print("--- Socio-economic ---"); display(real_world_tables["socioeconomic"])
print("--- Air quality ---"); display(real_world_tables["air_quality"])

--- Housing ---


,method,drop_pp,95% CI (pp),"drop_pp (original, 3 seeds)"
0,self_training,13.75,"[12.25, 15.23]",10.3
1,supervised_only,12.50,"[11.12, 14.15]",11.9
2,reweighted_self_training,12.22,"[10.55, 13.55]",9.7
3,label_propagation,4.27,"[2.28, 6.07]",2.9


--- Socio-economic ---


,method,drop_pp,95% CI (pp),"drop_pp (original, 3 seeds)"
0,supervised_only,-2.55,"[-3.60, -1.57]",-1.8
1,self_training,-2.78,"[-4.03, -1.57]",-1.5
2,label_propagation,-2.98,"[-4.68, -1.32]",-1.3
3,reweighted_self_training,-3.47,"[-5.08, -2.17]",-2.3


--- Air quality ---


,method,drop_pp,95% CI (pp),"drop_pp (original, 3 seeds)"
0,label_propagation,13.50,"[9.43, 18.36]",29.0
1,supervised_only,9.29,"[5.92, 12.80]",18.1
2,self_training,5.14,"[0.42, 10.53]",17.1
3,reweighted_self_training,4.14,"[-0.37, 9.29]",21.0


## 4. Pooled divergence-vs-accuracy correlation (Table 2), 20 seeds + honest CIs

Produced by `ssl_spatial.experiments.localized_mmd_analysis_v2`, which re-runs the original's exact divergence computation and merge logic on the 20-seed sweep, then wraps a seed-cluster bootstrap CI around the same pooled Pearson correlation the original reported as a bare point estimate.

In [5]:
corr = pd.read_csv(REPO_ROOT / "results" / "localized_mmd_comparison_v2_correlation_ci.csv")
corr["95% CI"] = corr.apply(lambda r: f"[{r['ci_low']:.3f}, {r['ci_high']:.3f}]", axis=1)
corr["r (original, 3 seeds)"] = corr["metric"].map({
    "kl_divergence": -0.67, "mmd": -0.61, "wasserstein": -0.55,
    "localized_mmd": -0.19, "kernel_weighted_local_mmd": -0.50,
})
corr[["metric", "r", "95% CI", "r (original, 3 seeds)"]].round(3)

,metric,r,95% CI,"r (original, 3 seeds)"
0,kl_divergence,-0.524,"[-0.601, -0.421]",-0.67
1,wasserstein,-0.454,"[-0.532, -0.351]",-0.55
2,mmd,-0.475,"[-0.556, -0.378]",-0.61
3,localized_mmd,-0.282,"[-0.348, -0.203]",-0.19
4,kernel_weighted_local_mmd,-0.396,"[-0.482, -0.326]",-0.50
